# A Spiking Model of Spinal Dorsal Horn Pain Processing

This notebook walks through a small spiking neural network model of the spinal dorsal horn -- the first relay station in the central nervous system where peripheral pain ("nociceptive") signals are processed before being sent to the brain.

The network has three Izhikevich neuron populations:

- **NS (nociceptor-specific)** neurons receive the incoming pain stimulus directly.
- **WDR (wide-dynamic-range)** neurons integrate input from other populations and are thought, in real dorsal horn physiology, to be the main encoders of perceived pain intensity.
- **INH (inhibitory interneurons)** provide feedback/feedforward inhibition -- a simplified stand-in for local and descending inhibitory control of pain.

Neurons are connected based on distance in a simple 2D spatial layout, with delayed, exponentially-decaying synaptic currents (fast excitatory / slower inhibitory kinetics, loosely AMPA/GABA-like).

This is a Python/NumPy port of an iterative series of MATLAB prototypes (see `../original_matlab/`). See `../README.md` for the full writeup, including a few genuine bugs in the original code that this port fixes rather than reproduces.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # so `import dorsal_horn_model` works from notebooks/

import numpy as np
import matplotlib.pyplot as plt

from dorsal_horn_model import (
    SimulationConfig,
    simulate_network,
    simulate_reference_loop,
    plot_raster,
    plot_synchrony,
    plot_weight_sweep,
    plot_synchrony_overlay,
)

## 1. Run a single simulation

`SimulationConfig` holds every tunable parameter (network size, timing, noise, stimulus). The defaults match the original MATLAB script: 40 NS neurons, 160 WDR neurons, 40 inhibitory interneurons, 2000 ms of simulated activity at a 0.1 ms time step.

In [ ]:
cfg = SimulationConfig(T=1000.0)  # shorter than the full 2000ms experiment, for a quick interactive run
rng = np.random.default_rng(42)

result = simulate_network(cfg, weight_factor=1.5, rng=rng)
print(f'{result.spike_record.sum()} total spikes across {cfg.N_total} neurons over {cfg.T} ms')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_raster(result, ax=axes[0])
axes[0].set_title('Raster plot (weight factor = 1.5)')
plot_synchrony(result, ax=axes[1])
axes[1].set_title('Synchrony (weight factor = 1.5)')
fig.tight_layout()

Red = NS, blue = WDR, green = INH (see `plot_raster`'s legend). You should see the NS population respond first to the stimulus, with activity propagating into WDR and INH over the following tens of milliseconds.

## 2. Fast vs. slow: verifying the vectorized simulation

The original MATLAB computed synaptic input with a nested `for i / for j` loop over every neuron pair, every time step -- `simulate_reference_loop()` is a direct, line-by-line translation of that. `simulate_network()` computes the identical mathematics with NumPy broadcasting instead, which is what makes the full 240-neuron / 20,000-step experiment below tractable (~20s instead of hours).

On a small network, both implementations should be driven by the same random draws and should agree exactly:

In [ ]:
small_cfg = SimulationConfig(N_NS=5, N_WDR=10, N_INH=5, dt=0.5, T=100.0, stimulus_window=(10, 20))

ref = simulate_reference_loop(small_cfg, weight_factor=1.5, rng=np.random.default_rng(7))
vec = simulate_network(small_cfg, weight_factor=1.5, rng=np.random.default_rng(7))

print('reference loop spikes:', int(ref.spike_record.sum()))
print('vectorized     spikes:', int(vec.spike_record.sum()))
print('identical spike trains:', bool(np.array_equal(ref.spike_record, vec.spike_record)))

(`simulate_reference_loop` and `simulate_network` each build their own network from the RNG they're given, so this compares two *independently constructed* small networks with the same seed -- see `validate_reference.py` at the project root for a stricter check that shares one network between the two implementations, run over multiple seeds.)

## 3. The synaptic weight sweep experiment

The main experiment in the original script scales synaptic strength by a factor (0.5x-2.0x) across otherwise-identical networks and looks at how population synchrony changes -- a simplified way of asking what happens to dorsal horn network activity as synaptic gain increases, one of the mechanisms proposed for central sensitization in chronic pain.

**Note:** in the original MATLAB, this sweep was a no-op due to a bug (the weight-scaling variable was applied to conductance matrices that were never actually used in the simulation loop). This port fixes that -- see the README for details. The effect below is the *fixed* behavior.

This runs the full-scale network (240 neurons, 2000ms) at four weight factors -- takes roughly 1-2 minutes.

In [ ]:
full_cfg = SimulationConfig(T=2000.0)
weight_factors = [0.5, 1.0, 1.5, 2.0]

results = []
for i, wf in enumerate(weight_factors):
    rng = np.random.default_rng(100 + i)
    res = simulate_network(full_cfg, weight_factor=wf, rng=rng)
    print(f'weight_factor={wf:>4.1f}  {int(res.spike_record.sum()):>7d} spikes')
    results.append(res)

In [ ]:
fig = plot_weight_sweep(results, weight_factors)

In [ ]:
fig2 = plot_synchrony_overlay(results, weight_factors)

## 4. Takeaways

- Low weight factors (0.5x-1.0x) barely sustain activity past the initial stimulus response.
- At 1.5x, the network settles into irregular, moderate-synchrony bursting.
- At 2.0x, the network is pushed into strong, sustained synchronous bursting across most of the WDR and inhibitory populations -- a caricature of the kind of runaway excitability implicated in central sensitization.

See `../README.md` for the full list of bugs found and fixed while porting this model, and `../validate_reference.py` for the correctness check between the fast and slow implementations.